# 01 - Data Collection

## Doel
Historische goudprijzen en macro-economische indicatoren ophalen via Yahoo Finance.

## Data bronnen
| Bron | Ticker | Variabele | Reden |
|------|--------|-----------|-------|
| Yahoo Finance | GC=F | Goud spot prijs (USD/oz) | Directe goudmarktprijs |
| Yahoo Finance | EURUSD=X | EUR/USD wisselkoers | Goud wordt in USD verhandeld; conversie naar EUR nodig |
| Yahoo Finance | GLD | Goud ETF (sentiment) | Marktsentiment indicator |

## Waarom deze variabelen?
Goudprijzen worden gedreven door 4 factoren:
1. **USD sterkte** - goud is omgekeerd gecorreleerd met USD
2. **Inflatie** - goud is een inflatiebuffer
3. **Rentevoeten** - hogere rente verhoogt de alternatieve kost van goud
4. **Marktsentiment** - ETF flows tonen institutionele vraag

In [ ]:
# Cel 1: Inladen van bibliotheken en instellen van paden

import sys
sys.path.insert(0, '..')

import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data_loader import (
    fetch_gold_price, fetch_eur_usd, fetch_gold_etf,
    fetch_all, convert_gold_to_eur_per_gram, save_raw
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
sns.set_palette('husl')

RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)
print('Setup complete.')

## 1. Goudprijs ophalen (GC=F - Gold Futures)

In [ ]:
# Cel 2: Goudprijs ophalen (GC=F - Gold Futures)

gold = fetch_gold_price(start='2020-01-01')
gold.head(10)

In [ ]:
# Cel 3: Goudprijs ophalen (GC=F - Gold Futures)

gold['gold_usd_oz'].plot(title='Gold Spot Price (USD/oz) - 2020 tot nu')
plt.ylabel('USD per troy ounce')
plt.xlabel('Datum')
plt.tight_layout()
plt.show()

## 2. EUR/USD wisselkoers ophalen

In [ ]:
# Cel 4: EUR/USD wisselkoers ophalen

eurusd = fetch_eur_usd(start='2020-01-01')
eurusd.head(10)

In [ ]:
# Cel 5: EUR/USD wisselkoers ophalen

eurusd['eur_usd'].plot(title='EUR/USD Exchange Rate - 2020 tot nu', color='green')
plt.ylabel('EUR/USD')
plt.xlabel('Datum')
plt.tight_layout()
plt.show()

## 3. GLD ETF ophalen (marktsentiment)

In [ ]:
# Cel 6: GLD ETF ophalen (marktsentiment)

etf = fetch_gold_etf(start='2020-01-01')
etf.head(10)

In [ ]:
# Cel 7: GLD ETF ophalen (marktsentiment)

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(etf.index, etf['gld_close'], color='gold', label='GLD Close')
ax1.set_ylabel('GLD Price (USD)', color='gold')
ax1.tick_params(axis='y', labelcolor='gold')

ax2 = ax1.twinx()
ax2.bar(etf.index, etf['gld_volume'], alpha=0.3, color='gray', label='Volume')
ax2.set_ylabel('Volume', color='gray')
ax2.tick_params(axis='y', labelcolor='gray')

plt.title('GLD ETF - Price & Volume')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.show()

## 4. Alle data samenvoegen

In [ ]:
# Cel 8: Alle data samenvoegen

merged = fetch_all(start='2020-01-01')
merged = convert_gold_to_eur_per_gram(merged)
print(f'\nMerged dataset: {merged.shape[0]} rijen, {merged.shape[1]} kolommen')
merged.head(10)

In [ ]:
# Cel 9: Alle data samenvoegen

merged.info()
print('\n--- Statistische samenvatting ---')
merged.describe()

## 5. Rauwe data opslaan

In [ ]:
# Cel 10: Rauwe data opslaan

save_raw(merged, 'gold_macro_merged.csv')
save_raw(gold, 'gold_spot_daily.csv')
save_raw(eurusd, 'eur_usd_daily.csv')
save_raw(etf, 'gold_etf_daily.csv')

print('\nAlle data succesvol opgeslagen in data/raw/')

## Samenvatting

| Stap | Resultaat |
|------|----------|
| Goudprijs | ~1500+ handelsdagen opgehaald |
| EUR/USD | Dagelijkse wisselkoersen |
| GLD ETF | Prijs + volume data |
| Samengevoegd | Eén dataframe met alle variabelen |

**Volgende stap:** Notebook 02 - Data Profiling & Cleaning